In [ ]:
# import 구문은 제일 위에 모아두는 거 기억하시죠?

# API KEY 설정

# 검색어 설정

# url 및 헤더 설정

# api 요청 및 결과값 json 반환
# [힌트] json <-> 파이썬 dictionary 변환
#           import json
#           json.loads(json)

# DB 연결 객체 생성

# SQL 수행을 위한 커서 생성

# API를 통해 받아온 정보를 가지고 INSERT 수행

# 데이터베이스 트랜잭션 처리 (commit -> 반영)

# 커서 및 연결 객체 종료 (반납)

In [2]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install dotenv


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# import 구문은 제일 위에
import urllib.parse
import urllib.request
import json
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()


True

In [20]:
client_id = os.getenv('naver_client_id')
client_secret = os.getenv("naver_secret")

In [76]:
# 검색어 설정
searchText = urllib.parse.quote('야구')

In [79]:

url = "https://openapi.naver.com/v1/search/book.json?query=" + searchText + "&display=100"

# 네이버 API 요청 준비
request = urllib.request.Request(url)

# 인증 헤더 추가
request.add_header("X-Naver-Client-Id",client_id) 
request.add_header("X-Naver-Client-Secret",client_secret)



In [90]:
# api 요청 및 결과값 json 반환
# [힌트] json <-> 파이썬 dictionary 변환
#           import json
#           json.loads(json)

response = urllib.request.urlopen(request) # api 호출
response_body = response.read() #응답데이터 (byte)
response_body = json.loads(response_body) # JSON -> Dictionary

book_list = response_body['items'] # 책 데이터 목록
response_body



{'lastBuildDate': 'Tue, 30 Jun 2026 15:48:03 +0900',
 'total': 593,
 'start': 501,
 'display': 93,
 'items': [{'title': '프로야구 스카우팅 리포트 2017 (800만 야구팬과 함께해온 KBO 리그 가이드)',
   'link': 'https://search.shopping.naver.com/book/catalog/32496969841',
   'image': 'https://shopping-phinf.pstatic.net/main_3249696/32496969841.20260509072907.jpg',
   'author': '이종진^김다미^권혁웅^장연재^이수만',
   'discount': '0',
   'publisher': '알에이치코리아(RHK)',
   'pubdate': '20170315',
   'isbn': '9788925561271',
   'description': '지금껏 이렇게 재미있는 스카우팅 리포트는 없었다! \n‘야구 덕후’가 만든 진정한 야구책,\n작정하고 공들인 2017 완전판\n\n매년 야구 시즌이 다가오면 출간되기만을 손꼽아 기다리는 책, 프로야구 스카우팅 리포트. 그 시리즈가 2017 새 시즌을 맞이하여 신선한 변화를 꾀했다. ‘야구 덕후’를 집필진으로 모시기로 한 것. 진정한 야구 덕후를 수소문한 끝에 ‘덕력 넘치는’ 최고의 집필진이 꾸려졌다. 전문가보다 더 전문가스러운 이들이 팬심에 의한, 팬심을 위한 책을 만들기 위해 온힘을 쏟아 펴낸 것이 이번 『프로야구 스카우팅 리포트 2017 완전판』이다. \n\n철저한 데이터 분석과 야구에 대한 무한한 애정으로 가득한 이 책은, 어느 한쪽으로 치우치거나 기존 가치관에 휩쓸리지 않고 객관적인 입장을 고수한다. 새 시즌을 기다리는 야구팬들의 마음을 꿰뚫고 있기에 팬들이 궁금해 하는 것이 무엇인지, 팬들이 리포트에서 가장 기대하는 내용이 무엇인지 정확히 알고 그 핵심을 짚어냈다.'},
  {

In [91]:
total_count = response_body['total']
start_num  = 1
loop_count = total_count//100+1 #100개씩 끊어서 몇 번 반복할지 계산
book_list = [] 

In [92]:
for i in range(loop_count):
    url = "https://openapi.naver.com/v1/search/book.json?query=" + searchText + "&display=100&start=" + str(start_num)

    # 네이버 API 요청 준비
    request = urllib.request.Request(url)

    # 인증 헤더 추가
    request.add_header("X-Naver-Client-Id",client_id) 
    request.add_header("X-Naver-Client-Secret",client_secret)
    
    
    response = urllib.request.urlopen(request)  # api 호출
    response_body = response.read()             #응답데이터 (byte)
    response_body = json.loads(response_body)   #JSON -> Dictionary
    
    book_list += response_body['items'] #기존 리스트에 이어붙임
    start_num += 100
    
    # 최대 1000건만 요청
    if start_num > 1000:
        break
len(book_list)

593

In [96]:
# DB연결 객체 생성
from datetime import datetime


with mysql.connector.connect(
    host = "localhost",
    user = "capybara",
    password = "1234",
    database = "bookdb"
) as connection:
    
    # SQL 수행을 위한 커서 생성
    with connection.cursor() as cursor:
        sql = f"insert into naver_book"\
            " (book_title,book_image,author, publisher, isbn,book_description,pub_date)"\
            " values (%s,%s,%s,%s,%s,%s,%s)"
        for book_info in book_list:
            # pub_date = datetime.strptime(book_info['pubdate'], "%Y%m%d")
            
            pub_str = book_info.get('pubdate','')
            
            #pub_date 가 비어있지 않으면 date 형식으로 변환
            if pub_str:
                pub_date = datetime.strptime(pub_str,'%Y%m%d').date()
            else: #비어있으면 None 값
                pub_date = None
            values = (
                book_info['title'],
                book_info['image'],
                book_info['author'],
                book_info['publisher'],
                book_info['isbn'],
                book_info['description'],
                pub_date
            )
            # print(values)
            
            cursor.execute(sql, values) # insert 문 실행
        
        # 데이터 베이스 트랜잭션 처리 (commit -> 반영)
        connection.commit()
            